<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day2/ExerciseXP/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

Found existing installation: pydantic_core 2.46.4
Uninstalling pydantic_core-2.46.4:
  Successfully uninstalled pydantic_core-2.46.4
Found existing installation: pydantic 2.13.4
Uninstalling pydantic-2.13.4:
  Successfully uninstalled pydantic-2.13.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 1.4.8 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.26 which is incompatible.
wandb 0.28.0 requires pydantic<3,>=2.6, but you have pydantic 1.10.26 which is incompatible.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
weasel 1.0.0 requires pydantic>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
google-genai 2.10.0 requi

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
# Simulation sécurisée du dataset si le fichier n'est pas présent localement
try:
    pdf = pd.read_csv('labelled_newscatcher_dataset.csv', sep=';')
except Exception:
    # Génération d'un DataFrame de secours conforme pour exécuter le RAG
    print("⚠️ Fichier local manquant. Génération d'un jeu de données Newscatcher temporaire...")
    data_mock = {
        'title': [
            "SpaceX launches new Starlink satellites into orbit successfully",
            "Global markets face minor correction amid inflation reports",
            "New vaccine breakthrough announced by medical researchers in Boston",
            "NASA Mars rover discovers ancient lakebed organic structures",
            "Tech giant unveils next-generation AI processor for cloud compute",
            "Rare animal species spotted in the deep rainforests of Madagascar"
        ] + [f"Random news headline article number {i} regarding science and space tech" for i in range(1000)]
    }
    pdf = pd.DataFrame(data_mock)

# TODO: Remplacement par la logique d'identification si la colonne 'id' n'existe pas
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

display(pdf.head())

# TODO: Création d'un sous-ensemble maniable (1000 premières lignes)
pdf_subset = pdf.iloc[:1000].copy()
display(pdf_subset[['id', 'title']].head())


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# TODO: Création des exemples d'entraînement à partir du sous-ensemble
faiss_train_examples = [
    example_create_fn(row['id'], row['title']) for _, row in pdf_subset.iterrows()
]
display(faiss_train_examples[:2])

# Initialisation et calcul de la matrice d'embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
print(f"Nombre d'embeddings : {len(faiss_title_embedding)}, Dimensions : {len(faiss_title_embedding[0])}")


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)

# Normalisation géométrique pour calculer une similarité par produit scalaire (Inner Product) équivalente à la similarité cosinus
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)

index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f"Total des documents indexés dans FAISS : {index_content.ntotal}")

def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # TODO: Encodage et normalisation de la requête utilisateur
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)

    # TODO: Recherche par similarité top-K
    sims, ids = index_content.search(query_vector, k)

    # Extraction et association des métadonnées
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0][:len(results)]
    return results

display(search_content('space development', pdf_to_index, k=5))


In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # TODO: encode the query using the sentence transformer model
    # FIX: Passage de la requête dans une liste pour obtenir une forme bidimensionnelle (1, 384) au format float32
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)

    # Execution de la recherche par similarité (Inner Product) sur l'index FAISS
    sims, ids = index_content.search(query_vector, k)

    # Extraction des lignes correspondantes depuis le DataFrame (en utilisant ids[0])
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

# Test de la recherche sémantique avec le mot-clé 'animal' sur le top-5
display(search_content('animal', pdf_to_index, k=5))


## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

# Nettoyage de la collection si elle existe déjà dans le cache
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

# TODO: Initialisation de la collection locale ChromaDB
collection = chroma_client.create_collection(name=collection_name)

# Préparation des payload complexes pour l'insertion (Format listes de chaînes exigé)
chroma_documents = pdf_subset['title'].tolist()
chroma_ids = pdf_subset['id'].astype(str).tolist()
chroma_embeddings = faiss_title_embedding.tolist()

# TODO: Ajout des documents vectorisés dans ChromaDB
collection.add(
    embeddings=chroma_embeddings,
    documents=chroma_documents,
    ids=chroma_ids
)

# TODO: Exécution d'une requête sémantique sur l'index vectoriel
results = collection.query(
    query_embeddings=model.encode(["space development"]).tolist(),
    n_results=3
)
print(json.dumps(results, indent=2))


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
model_id = 'google/flan-t5-small'

# TODO: Initialisation de la pipeline d'ingénierie générative Text2Text
pipe = pipeline(
    task="text2text-generation",
    model=model_id,
    max_length=64,
    device=-1  # Forcer CPU pour la stabilité (Mettre à 0 si GPU T4 activé et disponible)
)

question = "What's the latest news on space development?"

# Extraction dynamique du contexte sémantique récupéré à l'exercice 4
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)

# Construction du prompt d'instruction pour éliminer les hallucinations du LLM
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

# Inférence et affichage du résultat généré
response = pipe(prompt)[0]['generated_text']
print("\n" + "="*60)
print("👉 RÉPONSE CRÉÉE PAR LE FLUX RAG NATIF :")
print("="*60)
print(response)
print("="*60)
